# Анализ набора данных "insurance.csv"

## 1. Введение

В этом ноутбуке:
- Проводим EDA (исследовательский анализ данных), строим графики (гистограммы, pairplot, heatmap, boxplot).
- Очищаем данные от выбросов (IQR + Z-score).
- Подготавливаем данные к моделям.
- Обучаем и сравниваем несколько моделей (Linear/LogisticRegression, DecisionTree, CatBoost, MLP).
- Сохраняем метрики, графики обучения.
- **Интегрируемся** с DVC, чтобы выстраивать пайплайн обработки данных.

## 1.1 Что такое пайплайны?
**Пайплайн** (или конвейер) – это последовательность шагов (стадий) обработки данных. Например:
1. Предобработка (очистка)
2. Анализ (EDA)
3. Обучение модели

Возможны и дополнительные стадии (например, валидация, деплой и т.д.). Каждый шаг может зависеть от результатов предыдущих. **DVC** (Data Version Control) помогает хранить версии данных и полуавтоматически перестраивать только те стадии, чьи входные данные изменились.

## 1.2 Почему DVC?
При работе с данными часто нужно:
- Отслеживать изменения в больших файлов (CSV, модельных весов) без перегрузки Git.
- Хранить и версионировать данные + модели.
- Легко воспроизводить эксперименты (какой код и с какими параметрами дал ту или иную метрику).

DVC решает эти задачи, предлагая простую систему "stages" (как Makefile), кеширование данных и возможность хранить большие файлы не в Git.


## 2. Импорт библиотек и вспомогательные функции

In [ ]:
!pip install pandas numpy seaborn matplotlib scikit-learn catboost

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
from datetime import datetime
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from catboost import CatBoostRegressor, CatBoostClassifier
from sklearn.metrics import (
    r2_score, mean_squared_error,
    accuracy_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor, MLPClassifier

sns.set_theme(color_codes=True)

def save_plot(filename):
    current_time = datetime.now().strftime('%Y-%m-%d_%H-%M')
    dir_name = f"graphs/{current_time}"
    os.makedirs(dir_name, exist_ok=True)
    plt.savefig(f"{dir_name}/{filename}", dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

## 3. Загрузка и первичный осмотр данных

In [ ]:
# Предположим, у нас уже есть raw/insurance.csv
df_original = pd.read_csv("raw/insurance.csv")
display(df_original.head())
print("\nРазмеры датасета:", df_original.shape)
df_original.info()
display(df_original.describe(include='all'))

### Удаление дубликатов и пропусков

In [ ]:
df_original = df_original.drop_duplicates()
df_original = df_original.dropna()
print("После удаления дубликатов и пропусков:", df_original.shape)

## 4. EDA: распределения, матрица корреляций, boxplots

In [ ]:
num_cols = ['age', 'bmi', 'children', 'charges']
cat_cols = ['sex', 'smoker', 'region']

# Гистограммы числовых признаков
plt.figure(figsize=(12, 8))
for i, col in enumerate(num_cols, 1):
    plt.subplot(2, 2, i)
    sns.histplot(df_original[col], kde=True)
    plt.title(f"Distribution of {col}")
plt.tight_layout()
save_plot("01_num_distributions.png")

# Pairplot
sns.pairplot(df_original[num_cols], diag_kind='kde', corner=True)
plt.suptitle("Pairplot (numeric features)", y=1.02)
save_plot("02_pairplot_numeric.png")

# Корреляционная матрица (smoker → 0/1)
df_corr = df_original.copy()
df_corr['smoker'] = df_corr['smoker'].map({'no': 0, 'yes': 1})
corr_matrix = df_corr[num_cols + ['smoker']].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='YlGnBu')
plt.title("Correlation Heatmap")
save_plot("03_correlation_heatmap.png")

# Boxplots charges vs категориальные
plt.figure(figsize=(8,5))
sns.boxplot(x='smoker', y='charges', data=df_original, hue='smoker', dodge=False)
plt.title("Charges by Smoker")
save_plot("04_boxplot_charges_by_smoker.png")

plt.figure(figsize=(8,5))
sns.boxplot(x='sex', y='charges', data=df_original, hue='sex', dodge=False)
plt.title("Charges by Sex")
save_plot("05_boxplot_charges_by_sex.png")

plt.figure(figsize=(8,5))
sns.boxplot(x='region', y='charges', data=df_original, hue='region', dodge=False)
plt.title("Charges by Region")
save_plot("06_boxplot_charges_by_region.png")

## 5. Очистка данных (IQR + Z-score)
Уберём выбросы по столбцу `charges`.

In [ ]:
def filter_iqr(data, column, multiplier=1.5):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    mask = (data[column] >= Q1 - multiplier * IQR) & (data[column] <= Q3 + multiplier * IQR)
    return data[mask]

def filter_z_score(data, column, threshold=3):
    mean_val = data[column].mean()
    std_val = data[column].std()
    z_score = (data[column] - mean_val) / std_val
    return data[abs(z_score) <= threshold]

In [ ]:
# Копируем df_original -> df (рабочая версия)
df = df_original.copy()
df = filter_iqr(df, 'charges', multiplier=1.5)
df = filter_z_score(df, 'charges', threshold=3)
print("После фильтрации:", df.shape)

plt.figure(figsize=(8,5))
sns.histplot(df['charges'], kde=True)
plt.title("Distribution of charges AFTER filtering")
save_plot("07_charges_distribution_after_filter.png")

## 6. Подготовка к моделям
Две задачи:
- **Регрессия** (предсказывать `charges`).
- **Классификация** (предсказывать `smoker`).

### 6.1 Регрессия (charges)

In [ ]:
def prepare_regression_data(df_):
    df_temp = df_.copy()
    df_temp = pd.get_dummies(df_temp, columns=['sex','smoker','region'], drop_first=True)
    X = df_temp.drop('charges', axis=1)
    y = df_temp['charges']
    return X, y

X_reg, y_reg = prepare_regression_data(df)
print("X_reg.shape:", X_reg.shape, "y_reg.shape:", y_reg.shape)

### 6.2 Классификация (smoker)

In [ ]:
def prepare_classification_data(df_):
    df_temp = df_.copy()
    df_temp['smoker'] = df_temp['smoker'].map({'no': 0, 'yes': 1})
    y = df_temp['smoker']
    df_temp = df_temp.drop('smoker', axis=1)
    df_temp = pd.get_dummies(df_temp, columns=['sex','region'], drop_first=True)
    X = df_temp
    return X, y

X_clf, y_clf = prepare_classification_data(df)
print("X_clf.shape:", X_clf.shape, "y_clf.shape:", y_clf.shape)

## 7. Обучение моделей
Разобьём данные на train/test. Затем обучим:
1. Линейную регрессию.
2. Логистическую регрессию.
3. Дерево решений (регрессия).
4. CatBoost (регрессор и классификатор).
5. **MLP** (нейронная сеть) для регрессии и классификации.


In [ ]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42
)
print("Train shape (reg):", X_train_reg.shape, "Test shape (reg):", X_test_reg.shape)
print("Train shape (clf):", X_train_clf.shape, "Test shape (clf):", X_test_clf.shape)

### 7.1 Линейная регрессия (charges)

In [ ]:
linreg = LinearRegression()
linreg.fit(X_train_reg, y_train_reg)
y_pred_lin = linreg.predict(X_test_reg)
r2_lin = r2_score(y_test_reg, y_pred_lin)
rmse_lin = mean_squared_error(y_test_reg, y_pred_lin, squared=False)
print(f"[LinearRegression] R2: {r2_lin:.3f}, RMSE: {rmse_lin:.2f}")

# График Факт vs Предсказание
plt.figure(figsize=(6,5))
plt.scatter(y_test_reg, y_pred_lin, alpha=0.7, color='blue')
plt.plot([y_test_reg.min(), y_test_reg.max()],[y_test_reg.min(), y_test_reg.max()], color='red', lw=2)
plt.title("LinearReg: Actual vs Predicted (charges)")
save_plot("08_linreg_actual_vs_pred.png")

### 7.2 Логистическая регрессия (smoker)

In [ ]:
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_clf, y_train_clf)
y_pred_log = logreg.predict(X_test_clf)
acc_log = accuracy_score(y_test_clf, y_pred_log)
f1_log = f1_score(y_test_clf, y_pred_log)
print(f"[LogisticRegression] Accuracy: {acc_log:.3f}, F1: {f1_log:.3f}")

cm = confusion_matrix(y_test_clf, y_pred_log)
ConfusionMatrixDisplay(cm, display_labels=["No Smoker", "Smoker"]).plot(cmap="Blues")
plt.title("LogReg - Confusion Matrix")
save_plot("09_logreg_confusion_matrix.png")

### 7.3 Дерево решений (charges)

In [ ]:
dt_reg = DecisionTreeRegressor(random_state=42, max_depth=5)
dt_reg.fit(X_train_reg, y_train_reg)
y_pred_dt = dt_reg.predict(X_test_reg)
r2_dt = r2_score(y_test_reg, y_pred_dt)
rmse_dt = mean_squared_error(y_test_reg, y_pred_dt, squared=False)
print(f"[DecisionTreeRegressor] R2: {r2_dt:.3f}, RMSE: {rmse_dt:.2f}")

importances = dt_reg.feature_importances_
feature_names = X_train_reg.columns
imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False)
plt.figure(figsize=(6,4))
sns.barplot(x='importance', y='feature', data=imp_df)
plt.title("Feature Importances - DecisionTreeRegressor")
save_plot("10_dtreg_feature_importances.png")
imp_df

### 7.4 CatBoost (регрессор и классификатор)

In [ ]:
# Регрессор
cbr = CatBoostRegressor(verbose=0, random_state=42)
cbr.fit(X_train_reg, y_train_reg)
y_pred_cbr = cbr.predict(X_test_reg)
r2_cbr = r2_score(y_test_reg, y_pred_cbr)
rmse_cbr = mean_squared_error(y_test_reg, y_pred_cbr, squared=False)
print(f"[CatBoostRegressor] R2: {r2_cbr:.3f}, RMSE: {rmse_cbr:.2f}")

# Классификатор
cbc = CatBoostClassifier(verbose=0, random_state=42)
cbc.fit(X_train_clf, y_train_clf)
y_pred_cbc = cbc.predict(X_test_clf)
acc_cbc = accuracy_score(y_test_clf, y_pred_cbc)
f1_cbc = f1_score(y_test_clf, y_pred_cbc)
print(f"[CatBoostClassifier] Accuracy: {acc_cbc:.3f}, F1: {f1_cbc:.3f}")
cm_cbc = confusion_matrix(y_test_clf, y_pred_cbc)
ConfusionMatrixDisplay(cm_cbc, display_labels=["No Smoker", "Smoker"]).plot(cmap="Oranges")
plt.title("CatBoost - Confusion Matrix")
save_plot("11_catboost_confusion_matrix.png")

### 7.5 MLP (нейронная сеть)
#### 7.5.1 MLP Regressor

In [ ]:
# Масштабируем
scaler_reg = StandardScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_reg.transform(X_test_reg)

mlp_reg = MLPRegressor(
    hidden_layer_sizes=(64, 64),
    max_iter=500,
    random_state=42
)
mlp_reg.fit(X_train_reg_scaled, y_train_reg)
y_pred_mlp_reg = mlp_reg.predict(X_test_reg_scaled)
r2_mlp_reg = r2_score(y_test_reg, y_pred_mlp_reg)
rmse_mlp_reg = mean_squared_error(y_test_reg, y_pred_mlp_reg, squared=False)
print(f"[MLPRegressor] R2: {r2_mlp_reg:.3f}, RMSE: {rmse_mlp_reg:.2f}")

# Кривая обучения (loss)
plt.figure(figsize=(6,4))
plt.plot(mlp_reg.loss_curve_, label='Training loss')
plt.title("MLPRegressor - Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
save_plot("12_mlp_regressor_loss_curve.png")

# Факт vs Предсказание
plt.figure(figsize=(5,5))
plt.scatter(y_test_reg, y_pred_mlp_reg, alpha=0.7)
plt.plot([y_test_reg.min(), y_test_reg.max()],[y_test_reg.min(), y_test_reg.max()], color='red')
plt.title("MLPRegressor: Actual vs Predicted")
save_plot("13_mlp_regressor_actual_vs_pred.png")

# Гистограмма весов входного слоя
weights_layer0 = mlp_reg.coefs_[0].flatten()
plt.figure(figsize=(6,4))
plt.hist(weights_layer0, bins=30, alpha=0.7, color='purple')
plt.title("Histogram of MLPRegressor (Layer 0) Weights")
plt.xlabel("Weight Value")
plt.ylabel("Count")
save_plot("14_mlp_regressor_weights_hist.png")

#### 7.5.2 MLP Classifier

In [ ]:
# Масштабируем
scaler_clf = StandardScaler()
X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf)
X_test_clf_scaled = scaler_clf.transform(X_test_clf)

mlp_clf = MLPClassifier(
    hidden_layer_sizes=(64, 64),
    max_iter=500,
    random_state=42
)
mlp_clf.fit(X_train_clf_scaled, y_train_clf)
y_pred_mlp_clf = mlp_clf.predict(X_test_clf_scaled)
acc_mlp_clf = accuracy_score(y_test_clf, y_pred_mlp_clf)
f1_mlp_clf = f1_score(y_test_clf, y_pred_mlp_clf)
print(f"[MLPClassifier] Accuracy: {acc_mlp_clf:.3f}, F1: {f1_mlp_clf:.3f}")

# Кривая обучения
plt.figure(figsize=(6,4))
plt.plot(mlp_clf.loss_curve_, label='Training loss')
plt.title("MLPClassifier - Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
save_plot("15_mlp_classifier_loss_curve.png")

# Confusion Matrix
cm_mlp = confusion_matrix(y_test_clf, y_pred_mlp_clf)
ConfusionMatrixDisplay(cm_mlp, display_labels=["No Smoker", "Smoker"]).plot(cmap="Purples")
plt.title("MLPClassifier - Confusion Matrix")
save_plot("16_mlp_classifier_confusion_matrix.png")

## 8. Сравнение итоговых результатов

In [ ]:
data_regression = {
    'Model': [
        'LinearRegression',
        'DecisionTreeRegressor',
        'CatBoostRegressor',
        'MLPRegressor'
    ],
    'R2': [
        round(r2_lin, 3),
        round(r2_dt, 3),
        round(r2_cbr, 3),
        round(r2_mlp_reg, 3)
    ],
    'RMSE': [
        round(rmse_lin, 2),
        round(rmse_dt, 2),
        round(rmse_cbr, 2),
        round(rmse_mlp_reg, 2)
    ]
}
df_reg_results = pd.DataFrame(data_regression)

data_classification = {
    'Model': [
        'LogisticRegression',
        'CatBoostClassifier',
        'MLPClassifier'
    ],
    'Accuracy': [
        round(acc_log, 3),
        round(acc_cbc, 3),
        round(acc_mlp_clf, 3)
    ],
    'F1': [
        round(f1_log, 3),
        round(f1_cbc, 3),
        round(f1_mlp_clf, 3)
    ]
}
df_clf_results = pd.DataFrame(data_classification)

print("=== Регрессия (charges) ===")
display(df_reg_results)

print("=== Классификация (smoker) ===")
display(df_clf_results)

## 9. DVC: Как мы настроили и зачем
1. **Инициализировали Git** (если не было): `git init`
2. **Инициализировали DVC**: `dvc init`
3. **Добавили датасет**: `dvc add raw/insurance.csv` (и закоммитили `.dvc` файл)
4. Создали **несколько стадий** (preprocessing, eda, train_reg, train_clf) в `dvc.yaml`.
5. Выполняем `dvc repro`, и DVC автоматически понимает, что нужно перезапустить, если изменился CSV или скрипты.
6. **Выгрузка пайплайна** в `.dot`:
   ```bash
   dvc dag --dot > pipeline.dot
   dot -Tpng pipeline.dot -o pipeline.png
   ```
   Теперь у вас есть `pipeline.png` – наглядная схема стадий.

Таким образом, мы **версионируем** данные, **храним** модели и **получаем** воспроизводимый пайплайн.


## 10. Выводы
1. Курение (`smoker`) – ключевой фактор, заметно повышающий расходы (`charges`).
2. Возраст (`age`) и ИМТ (`bmi`) также существенны.
3. IQR+Z-score фильтрация убрала выбросы, что улучшило стабильность.
4. Модели CatBoost и MLP часто дают лучшие метрики.
5. DVC позволяет хранить и пересобирать все шаги (EDA, очистка, обучение) при изменении кода или входных данных.
6. Для полноценного TensorBoard с графом слоёв рекомендуем Keras/PyTorch.
